# Computación Evolutiva - PC2

- Sergio Sebastian Pezo Jimenez
- 20224087G

In [ ]:
import numpy as np

In [ ]:
print("xd")

## Pregunta 1


In [ ]:
# Primero tenemos que calcular cuantos bits necesitamos
puntos = 2097152
l = 0
while 2**l < puntos:
    l += 1

# Individuos
n = 50

print("Bits:", l)

# a)
print("a) Número total de esquemas", 3**l)

# b)
print("b) Número total e puntos de búsqueda", 2**l)

# c)
print("c) Número de esquemas por individuo", 2**l)

# d)
print("d) Inferior:", 2**l, "Superior:", n*2**l)

# d) y e) Nos piden calcular una cotas, así que definiremos un esquema, para trabajar en base a el
# Individuos que pertenecen al esquema H
m = 10
# Aptitud promedio del esquema
fH = 12
# Aptitud de los otros 40 individuos
f_otros = 9.5
# Probabilidad de cruzamiento
pc = 0.25
# Probabilidad de mutacion
pm = 0.01

# Dos posiciones fijas: 11*******************
orden = 2
# Distancia entre las posiciones fijas
delta = 1

# Aptitud promedio de toda la poblacion n definida al inicio
f_prom = (m*fH + (n-m)*f_otros)/n

# Cota inferior por el teorema de esquemas
cota = m*(fH/f_prom)*(1-pc*delta/(l-1)-orden*pm)

print("e) Aptitud promedio:", f_prom)
print("e) Cota inferior esperada:", cota)

Bits: 21
a) Número total de esquemas 10460353203
b) Número total e puntos de búsqueda 2097152
c) Número de esquemas por individuo 2097152
d) Inferior: 2097152 Superior: 104857600
e) Aptitud promedio: 10.0
e) Cota inferior esperada: 11.61


# Pregunta 2

In [ ]:
# Definimos los limites x, y, z  de las variables x, y, z, respectivamente
limites = [(-20, 125), (0, 1200000), (-0.1, 1)]
precision = [0.5, 10000, 0.001]

# Calculamos la cantidad minima de bits por variable con la precisiones deseadas
longitudes = []
for i in range(3):
    a, b = limites[i]
    l = int(np.ceil(np.log2((b-a)/precision[i] + 1)))
    longitudes.append(l)

print("Bits:", longitudes)
print("El número mínimo de bits es:", sum(longitudes))

puntos = [(-20, 0, -0.1), (125, 1200000, 1), (50, 100000, 0.597)]

# Calculamos la cadena binaria para cada punto
for punto in puntos:
    codigos = []
    valores = []

    for i in range(3):
        a, b = limites[i]
        l = longitudes[i]

        # Convertimos el valor real a un entero
        entero = round((punto[i]-a)/(b-a)*(2**l-1))

        # Pasamos el entero a binario usando l bits
        codigos.append(format(entero, "0"+str(l)+"b"))

        # Decodificamos para comprobar la aproximacion
        valores.append(a+(b-a)*entero/(2**l-1))

    print("Punto:", punto)
    print("Binarios:", *codigos)
    print("Concatenado:", "".join(codigos))
    print("Decodificado:", valores)

Bits: [9, 7, 11]
El número mínimo de bits es: 27
Punto: (-20, 0, -0.1)
Binarios: 000000000 0000000 00000000000
Concatenado: 000000000000000000000000000
Decodificado: [-20.0, 0.0, -0.1]
Punto: (125, 1200000, 1)
Binarios: 111111111 1111111 11111111111
Concatenado: 111111111111111111111111111
Decodificado: [125.0, 1200000.0, 1.0]
Punto: (50, 100000, 0.597)
Binarios: 011110111 0001011 10100010001
Concatenado: 011110111000101110100010001
Decodificado: [50.088062622309195, 103937.00787401575, 0.596971177332682]


# Pregunta 3


Trabajaremos para el intervalo de [-5, 8]

In [ ]:
np.random.seed(7)


X_BOUND = [-5.0, 8.0]

# Definimos nuestros parametros para el algoritmo genetico
n = 2
bits = 12
pc = 0.25
pm = 0.01
generaciones = 150

def F(x):
    return 3*x**10


def get_fitness(valores):
    return valores + 1e-3

def translateDNA(pop):
    pesos = 2**np.arange(bits)[::-1]
    return X_BOUND[0] + (X_BOUND[1]-X_BOUND[0])*pop.dot(pesos)/(2**bits-1)

# Seleccion por ruleta
def select(pop, fitness):
    ps = fitness/fitness.sum()
    indices = np.random.choice(np.arange(n), size=n, replace=True, p=ps)
    return pop[indices]

# Cruzamiento de un punto entre dos cromosomas por corte
def crossover(padre, pop):
    if np.random.rand() < pc:
        j = np.random.randint(n)
        corte = np.random.randint(1, bits)
        padre[corte:] = pop[j, corte:]
    return padre

# Para la mutacion, simplemente cambiaremos un bit con probabilidad pm
def mutate(hijo):
    for j in range(bits):
        if np.random.rand() < pm:
            hijo[j] = 1 - hijo[j]
    return hijo

# Crear poblacion inicial
pop = np.random.randint(2, size=(n, bits))

for t in range(generaciones):
    # Evaluamos los individuos
    valores = F(translateDNA(pop))
    fitness = get_fitness(valores)

    # Guardamos el mejor cromosoma
    mejor = pop[np.argmax(valores)].copy()

    # Imprimimos el progreso cada 10 generaciones
    if t == 0 or (t+1) % 10 == 0:
        print("Generacion", t+1, "Mejor:", np.max(valores))

    # Seleccionamos y generamos descendientes
    pop = select(pop, fitness)
    copia = pop.copy()

    for i in range(n):
        hijo = crossover(pop[i].copy(), copia)
        pop[i] = mutate(hijo)

    # Conservamos al mejor individuo (elitismo)
    pop[0] = mejor

# Resultado final
x = translateDNA(pop)
valores = F(x)

print("Mejor x:", x[np.argmax(valores)])
print("Maximo f(x):", np.max(valores))


Generacion 1 Mejor: 2998394484.9638968
Generacion 10 Mejor: 3221225472.0
Generacion 20 Mejor: 3221225472.0
Generacion 30 Mejor: 3221225472.0
Generacion 40 Mejor: 3221225472.0
Generacion 50 Mejor: 3221225472.0
Generacion 60 Mejor: 3221225472.0
Generacion 70 Mejor: 3221225472.0
Generacion 80 Mejor: 3221225472.0
Generacion 90 Mejor: 3221225472.0
Generacion 100 Mejor: 3221225472.0
Generacion 110 Mejor: 3221225472.0
Generacion 120 Mejor: 3221225472.0
Generacion 130 Mejor: 3221225472.0
Generacion 140 Mejor: 3221225472.0
Generacion 150 Mejor: 3221225472.0
Mejor x: 8.0
Maximo f(x): 3221225472.0
